### Notebook for visualizing the embeddings
- need result folder from previous notebook
- dataframe from previous notebook

In [ ]:
from huggingface_hub import hf_hub_download
from datasets import load_dataset
import pandas as pd
from typing import Literal, Dict

import torch, os
import umap
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool


In [ ]:
# huggingface data load
dataset = load_dataset(
    "mogam-ai/ecoli-embeddings", 
    data_files="data_ecoli.csv"  # 정확한 파일명 사용
)
# transform huggingface dataset into pandas DataFrame
data_info = dataset['train'].to_pandas()

data = torch.load(hf_hub_download(repo_id="mogam-ai/ecoli-embeddings", 
                                  filename="ecoli_emb_with_eos.pt", repo_type="dataset"))

/fsx/tmp/jhhong/ipykernel_3612962/3395755768.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(hf_hub_download(repo_id="mogam-ai/ecoli-embeddings",


In [10]:
data_info
print(set(data_info['y']))

{0, 1, 2}


In [ ]:
class VizData:
    def __init__(
        self,
        info_df: pd.DataFrame = data_info,
        embedding_data: torch.Tensor = None,
        choice_for_embedding: Literal["mean", "max", "eos"] = "mean", 
        label_info: Dict = {'Low':0 ,'Medium':1,'High':2},
       ):

        self.info = info_df
        self.labels = info_df["y"].values
        self.label_info = label_info
        self.reverse_label_info = {v: k for k, v in label_info.items()}
        self.choice_for_embedding = choice_for_embedding

        self.umap_df = pd.DataFrame()
        self.umap_df["labels"] = self.labels

        # use user def colors
        colors = ['red','blue','green'] 
        markers = ['circle', 'square', 'triangle']
        sizes = [10, 10, 12, 14]
        self.umap_df['color'] = self.umap_df['labels'].map(lambda x: colors[x])
        self.umap_df['marker'] = self.umap_df['labels'].map(lambda x: markers[x])
        self.umap_df['legend'] = self.umap_df['labels'].map(self.reverse_label_info)
        self.umap_df['size'] = self.umap_df['labels'].map(lambda x: sizes[x])
        
        self.embedding_data = embedding_data
        self.embeddings = self.get_embedding(choice_for_embedding, self.embedding_data)

        print(f"Loaded embedding with shape {self.embeddings.shape}.")


    @staticmethod
    def get_embedding(choice, embedding_data):
        match choice:
            case "mean":
                embeddings = embedding_data.mean(dim=1).cpu()
            case "max":
                embeddings = embedding_data.max(dim=1) 
            case _:
                raise ValueError("Invalid choice for embedding. Choose 'mean', 'max', or 'eos'.")
        return embeddings
    

In [17]:
viz = VizData(
    info_df=data_info,
    embedding_data=data,
    choice_for_embedding="mean",
)
 
    

Loaded embedding with shape torch.Size([1500, 768]).


In [18]:
viz.embeddings
print(viz.embeddings.shape)

torch.Size([1500, 768])


In [19]:
# Visualization

umap_model = umap.UMAP(
    n_neighbors=20,
    n_epochs=1500,
    min_dist=1.0,
    init='spectral',
    repulsion_strength=0,
    # metric='euclidean',
    metric = 'manhattan',
    n_components=2,
    random_state=42,
    spread=1.0
)
umap_embeddings = umap_model.fit_transform(viz.embeddings)
umap_df = viz.umap_df

# add umap embeddings to the dataframe
umap_df["x"] = umap_embeddings[:, 0]
umap_df["y"] = umap_embeddings[:, 1]

# Map colors to labels
# Step 5: Create Bokeh plot with tooltips
source = ColumnDataSource(umap_df)

hover = HoverTool(tooltips=[
    ("Label", "@legend"),
    ('Marker', '@marker'),
    ('Size','@size'),
])

plot = figure(
    title="UMAP for RNA CodonBERT Embeddings",
    tools=["pan,wheel_zoom,reset", hover],
    width=700, height=600
)
# 플롯의 전체 테두리 설정
plot.outline_line_color = "black"
plot.outline_line_width = 2

plot.scatter(
    'x', 'y', 
    source=source, 
    color='color',
    marker = 'marker',
    size= 'size',
    legend_field='legend',
    alpha=0.8,
    line_color = 'white',
    line_width = 1.5
)

plot.xaxis.axis_label = 'UMAP Dimension 1'
plot.yaxis.axis_label = 'UMAP Dimension 2'
plot.legend.location = 'bottom_right'
# x축과 y축의 눈금 및 숫자 제거
plot.xaxis.visible = False
plot.yaxis.visible = False
plot.xgrid.visible = False
plot.ygrid.visible = False
show(plot)



Loading BokehJS ...

/fsx/home/jhhong/miniconda3/envs/rna/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
